# Step By Step 04 - Factor Preprocessing
--------------

Factors generally require preprocessing, principally standardization and winsorization, before being used to generate signals.

In [ ]:
import asyncio
import datetime as dt


In [ ]:
# First, prepare the raw data
import numpy as np
import pyarrow as pa


## 1. Standardization

In [ ]:
import pyarrow.compute as pc

rng = np.random.default_rng(42)


## 2. Winsorization

In [ ]:
n_dates = 6
n_assets = 12


## 3. Neutralization

This step is particularly characteristic of quantitative research. For a factor model, the original factor is normally neutralized against **risk factors**, retaining only the residual component.

In [ ]:
dates = [dt.datetime(2026, 1, 5) + dt.timedelta(days=index) for index in range(n_dates)]
codes = np.array([f"{index:06d}" for index in range(1, n_assets + 1)])


In [ ]:
n_rows = n_dates * n_assets
panel = pa.table({
    "trade_date": [value for value in dates for _ in range(n_assets)],
    "code": np.tile(codes, n_dates),
    "factor": rng.normal(size=n_rows),
    "value": rng.normal(size=n_rows),
    "dx": rng.normal(0.0, 0.01, size=n_rows),
    "industry": np.tile(np.array(["Bank", "Auto", "Health"]), n_dates * n_assets // 3),
    "benchmark_weight": np.full(n_rows, 1.0 / n_assets),
    "BETA": np.tile(np.linspace(0.8, 1.2, n_assets) ** 2, n_dates),
    "SIZE": np.tile(np.linspace(-1.0, 1.0, n_assets), n_dates),
})
from alphamind.data.processing import factor_processing


## 4. Pipeline

**alpha-mind** provides a unified processing function so users can combine multiple preprocessing steps in one pipeline.

In [ ]:
from alphamind.data.standardize import standardize
from alphamind.data.winsorize import winsorize_normal


In [ ]:
values = np.column_stack([
    panel["factor"].to_numpy(zero_copy_only=False),
    panel["value"].to_numpy(zero_copy_only=False),
])
groups = np.unique(panel["industry"].to_pylist(), return_inverse=True)[1]


In [ ]:
risk = panel["SIZE"].to_numpy(zero_copy_only=False).reshape(-1, 1)
processed = factor_processing(
    values,
    pre_process=[winsorize_normal, standardize],
    risk_factors=risk,
    post_process=[standardize],
    groups=groups,
)


In [ ]:
assert np.isfinite(processed).all()
pa.table({"code": panel["code"], "factor": processed[:, 0], "value": processed[:, 1]})
